# Mine GN-TRVN hard negatives

GreenNode-Table-Markdown-Retrieval-VN ships train qrels and all query texts
(root `queries.jsonl`) but no negatives. This notebook pairs every train query
with its positive table document, mines `NUM_NEGATIVES` hard negatives with a
dense model (ANCE-style, as in the GN-TRVN paper) and writes
`OUTPUT_DIR/train.parquet` with columns `query, positive, negative_1 .. negative_n`,
the format `legal_st.data.load_triplet_records` reads
(`train_dataset: data/gn-trvn-hard-negatives` in `configs/gn-trvn-*.yaml`).

Only train documents are used as the mining corpus, so no test document can leak
into training.

**Expected output size**

- Input: 143,106 train qrels (one positive table per query) over 35,742 train tables.
- Output: at most one row per train qrel, i.e. **at most 143,106 rows**. A query is
  dropped when fewer than `NUM_NEGATIVES` candidates pass the relative-margin
  filter; the final count is printed in the "Save and report size" cell.
- Each row holds 1 query (~88 chars) plus `1 + NUM_NEGATIVES` tables (~1,264 chars
  each), so with 3 negatives that is roughly 730 MB of raw text before parquet
  compression.

Run it locally from the `notebooks/` folder in the `legal-st-fa2` environment
(`environment-fa2.yml`), or on Kaggle (GPU T4 x2, Internet on): the setup cell
then installs the pinned libraries, clones this repo into `/kaggle/working`
and encodes on both GPUs. Push your latest changes to `REPO_URL` first, because
Kaggle runs the cloned code.

In [ ]:
# Parameters
import os

ON_KAGGLE = os.path.isdir("/kaggle/working")
WORK_DIR = "/kaggle/working" if ON_KAGGLE else ".."

MINING_MODEL = "BAAI/bge-m3"          # dense model used to rank candidate negatives
OUTPUT_DIR = f"{WORK_DIR}/data/gn-trvn-hard-negatives"
CACHE_FOLDER = f"{WORK_DIR}/cache/gn-trvn-embeddings"  # reused if the session restarts
NUM_NEGATIVES = 3                     # the paper used ~7
RANGE_MIN = 0                         # skip the top-N most similar candidates
RANGE_MAX = None                      # only consider the top-N candidates (None: decided by ST)
RELATIVE_MARGIN = 0.05                # keep negatives scoring below 0.95 * positive score
SAMPLING_STRATEGY = "top"             # "top" or "random"
MAX_SEQ_LENGTH = 1024                 # GN-TRVN tables are at most ~750 tokens
BATCH_SIZE = 32                       # per GPU; fits bge-m3 fp16 at 1024 tokens on a 16 GB T4
USE_MULTI_GPU = True                  # one encode process per GPU when more than one is visible
USE_FAISS = False
ATTN_IMPLEMENTATION = "auto"          # T4 has no FlashAttention-2, so this resolves to sdpa
LIMIT_QUERIES = None                  # e.g. 2000 for a quick run
HUB_REPO_ID = None                    # e.g. "your-name/gn-trvn-hard-negatives" to push the result
SEED = 42

# Kaggle only: where to get this repo's code.
REPO_URL = "https://github.com/QuangDiy/legal-st"
REPO_BRANCH = "main"

## Kaggle setup

Installs the same library versions as `environment-fa2.yml` on top of Kaggle's
preinstalled PyTorch and clones the repo. Does nothing outside Kaggle.

In [ ]:
import subprocess
import sys

if ON_KAGGLE:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "sentence-transformers==5.1.2", "transformers==4.57.1",
         "datasets>=2.19,<4", "huggingface-hub>=0.29,<1.0"],
        check=True,
    )
    REPO_DIR = f"{WORK_DIR}/legal-st"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR],
            check=True,
        )
    else:  # refresh a clone left by an earlier run so newly pushed code is used
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"], check=True)
    print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"],
                         capture_output=True, text=True).stdout)
else:
    REPO_DIR = ".."

# Use the repo's src/ when legal_st is not pip-installed.
SRC = os.path.abspath(os.path.join(REPO_DIR, "src"))
if os.path.isdir(SRC) and SRC not in sys.path:
    sys.path.insert(0, SRC)

## Log in to the Hugging Face Hub

Needed for `HUB_REPO_ID` pushes and avoids anonymous download rate limits.
The token is read from the `HF_TOKEN` environment variable, then Kaggle or Colab
secrets named `HF_TOKEN`; otherwise an interactive login prompt is shown.
Never paste the token into this notebook.

In [ ]:
import os

from huggingface_hub import login, notebook_login

os.environ["HF_TOKEN"] = "hf_xxxxxxxxxxxxxxxxxxxxxxxxx"


def find_hf_token() -> str | None:
    if os.getenv("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    try:  # Kaggle
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:  # Colab
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None


hf_token = find_hf_token()
if hf_token:
    login(token=hf_token)
else:
    notebook_login()

In [ ]:
import json

import torch
from datasets import Dataset, load_dataset
from huggingface_hub import hf_hub_download
from sentence_transformers.util import mine_hard_negatives
from tqdm.auto import tqdm

from legal_st.modeling import load_sentence_transformer
from legal_st.utils import ensure_dir, normalize_text, safe_max_seq_length, save_json, set_seed

GN_TRVN = "GreenNode/GreenNode-Table-Markdown-Retrieval-VN"
set_seed(SEED)

## Load train (query, positive) pairs

The corpus `title` column is the literal string "None" for every row, so only
`text` is used.

In [ ]:
def read_jsonl(repo_file: str) -> list[dict]:
    path = hf_hub_download(GN_TRVN, repo_file, repo_type="dataset")
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


qrels = [row for row in read_jsonl("qrels/train.jsonl") if int(row["score"]) > 0]
if LIMIT_QUERIES is not None:
    qrels = qrels[:LIMIT_QUERIES]

queries = {row["_id"]: row["text"] for row in read_jsonl("queries.jsonl")}
corpus_rows = load_dataset(GN_TRVN, "corpus", split="test")
corpus = dict(zip(corpus_rows["id"], corpus_rows["text"]))

pairs = {"query": [], "positive": []}
for row in tqdm(qrels, desc="Pairing queries with tables"):
    query_text = queries.get(row["query-id"])
    doc_text = corpus.get(row["corpus-id"])
    if query_text and doc_text:
        pairs["query"].append(normalize_text(query_text))
        pairs["positive"].append(normalize_text(doc_text))
pairs = Dataset.from_dict(pairs)

print(f"Train qrels: {len(qrels):,}")
print(f"Train pairs: {len(pairs):,} ({len(set(pairs['positive'])):,} unique positive tables, "
      f"{len(set(pairs['query'])):,} unique query texts)")

## Load the mining model

In [ ]:
model = load_sentence_transformer(MINING_MODEL, ATTN_IMPLEMENTATION)
if torch.cuda.is_available():
    model.half()
model.max_seq_length = min(MAX_SEQ_LENGTH, safe_max_seq_length(model))

# One encode process per GPU (Kaggle T4 x2 -> cuda:0, cuda:1).
gpu_count = torch.cuda.device_count()
multi_process = [f"cuda:{i}" for i in range(gpu_count)] if USE_MULTI_GPU and gpu_count > 1 else False
print(f"Mining model: {MINING_MODEL} (max_seq_length={model.max_seq_length}, device={model.device})")
print(f"Encode devices: {multi_process or model.device}")

## Mine hard negatives

`corpus=None` makes the candidates the train positives only (no test leakage).

In [ ]:
mined = mine_hard_negatives(
    pairs,
    model,
    anchor_column_name="query",
    positive_column_name="positive",
    range_min=RANGE_MIN,
    range_max=RANGE_MAX,
    relative_margin=RELATIVE_MARGIN,
    num_negatives=NUM_NEGATIVES,
    sampling_strategy=SAMPLING_STRATEGY,
    output_format="n-tuple",
    batch_size=BATCH_SIZE,
    use_faiss=USE_FAISS,
    use_multi_process=multi_process,
    cache_folder=CACHE_FOLDER,  # skip re-encoding after a restart with the same inputs
    verbose=True,  # tqdm bars with ETA for query/table encoding
)

# Rename by position so the output never depends on ST's column naming.
columns = ["query", "positive"] + [f"negative_{i}" for i in range(1, len(mined.column_names) - 1)]
mined = mined.rename_columns(dict(zip(mined.column_names, columns)))

## Save and report size

In [ ]:
output_dir = ensure_dir(OUTPUT_DIR)
output_path = output_dir / "train.parquet"
mined.to_parquet(str(output_path))

text_chars = sum(len(text) for name in tqdm(columns, desc="Measuring text size") for text in mined[name])
stats = {
    "pairs": len(pairs),
    "rows": len(mined),
    "dropped": len(pairs) - len(mined),
    "kept_ratio": round(len(mined) / max(len(pairs), 1), 4),
    "texts_per_row": len(columns),
    "raw_text_mb": round(text_chars / 1e6, 1),
    "parquet_mb": round(output_path.stat().st_size / 1e6, 1),
}
save_json(
    {
        "mining_model": MINING_MODEL,
        "num_negatives": NUM_NEGATIVES,
        "range_min": RANGE_MIN,
        "range_max": RANGE_MAX,
        "relative_margin": RELATIVE_MARGIN,
        "sampling_strategy": SAMPLING_STRATEGY,
        "max_seq_length": MAX_SEQ_LENGTH,
        "limit_queries": LIMIT_QUERIES,
        "seed": SEED,
        "columns": columns,
        **stats,
    },
    output_dir / "mining_args.json",
)
print(json.dumps(stats, indent=2))
print(f"Saved: {output_path}")

In [ ]:
sample = mined[0]
for name in columns:
    print(f"--- {name} ---\n{sample[name][:300]}\n")

## Optional: push to the Hugging Face Hub

In [ ]:
if HUB_REPO_ID:
    mined.push_to_hub(HUB_REPO_ID, split="train")
    print(f"Pushed to: https://huggingface.co/datasets/{HUB_REPO_ID}")